In [0]:
%pip install sqlglot -q
%restart_python

In [0]:
import sqlglot
import re
from sqlglot.expressions import Table, CTE
from sqlglot.errors import ErrorLevel
from pyspark.sql.functions import udf, col, explode, split, when, size, element_at, trim, upper
from pyspark.sql.types import ArrayType, StringType

def clean_bo_sql(sql):
    """Remove SAP BO-specific syntax that sqlglot cannot parse."""
    sql = re.sub(r"@Prompt\((?:[^()]*|\([^()]*\))*\)", "'PROMPT_PLACEHOLDER'", sql)
    sql = re.sub(r"@Variable\((?:[^()]*|\([^()]*\))*\)", "'VAR_PLACEHOLDER'", sql)
    sql = re.sub(r'\.\s+', '.', sql)
    return sql

def extract_tables(sql):
    try:
        clean_sql = clean_bo_sql(sql or "")
        tables = set()
        cte_names = set()
        for parsed in sqlglot.parse(clean_sql, read="oracle", error_level=ErrorLevel.IGNORE):
            for cte in parsed.find_all(CTE):
                cte_names.add(cte.alias.lower())
            for t in parsed.find_all(Table):
                catalog = t.catalog
                schema = t.db
                name = t.name
                full_name = ".".join(
                    part for part in [catalog, schema, name] if part
                )
                if full_name.lower() not in cte_names:
                    tables.add(full_name)
        return sorted(tables)
    except Exception as e:
        return []

extract_tables_udf = udf(extract_tables, ArrayType(StringType()))

# Step 1: Get unprocessed Document_Ids using Python set difference (no ANTI JOIN)
# Query 1: Already-processed IDs (~7K distinct values from 170K rows)
processed_ids = set(    row.Document_Id for row in
    spark.sql("SELECT DISTINCT Document_Id FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source").collect())
print(f"Already processed: {len(processed_ids)} documents")

# Query 2: All candidate IDs from source (excluding non-SQL rows)
candidate_ids = [ row.Document_Id for row in
    spark.sql("""
        SELECT DISTINCT Document_Id 
        FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
        WHERE SQL_Query NOT IN ('Error retrieving Query Plan', 'Data Source Type excel not handled for SQL extraction')
    """).collect()]
print(f"Total candidates: {len(candidate_ids)} documents")

# Set difference in Python (instant) — no expensive SQL join
document_ids = [doc_id for doc_id in candidate_ids if doc_id not in processed_ids][:1000]
print(f"Documents to process: {len(document_ids)}")

if not document_ids:
    print("All documents processed!")
else:
    # Step 2: Filter source table
    df_source = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms")
    df_filtered = df_source.filter(col("Document_Id").isin(document_ids))

    # Step 3: Apply UDF once across all filtered rows
    df_with_tables = df_filtered.withColumn("parsed_tables", extract_tables_udf(col("SQL_Query")))
    df_exploded = df_with_tables.withColumn("table", explode(col("parsed_tables")))

    # Step 4: Split and uppercase
    split_col = split(col("table"), r"\.")
    df_final = df_exploded.withColumn(
        "sql_table", upper(trim(col("table")))
    ).withColumn(
        "table_Name", upper(trim(element_at(split_col, -1)))
    ).withColumn(
        "schema_Name", when(size(split_col) > 1, upper(trim(element_at(split_col, -2))))
    )

    # Step 5: Select target schema and write
    final_df = df_final.select(
        col("Document_Id"),
        col("Document_CUID"),
        col("Document_name"),
        col("Full_path"),
        col("lastAuthor").alias("updated_by"),
        col("Connection_Name").alias("source_DB_connection"),
        col("sql_table"),
        col("table_Name"),
        col("schema_Name")
    ).distinct()

    final_df.write.mode("append").saveAsTable(
        "dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source"
    )
    print("Done. Batch written to active_webi_source.")

    # Step 6: Compact small files
    spark.sql("OPTIMIZE dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source")
    print("Table optimized.")

In [0]:
import sqlglot
import re
from sqlglot.expressions import Table, CTE
from pyspark.sql.functions import udf, col, explode, split, when, size, element_at, trim, upper
from pyspark.sql.types import ArrayType, StringType

def clean_bo_sql(sql):
    """Remove SAP BO-specific syntax that sqlglot cannot parse."""
    # Replace @Prompt(...) — handles nested parens like value(s)
    sql = re.sub(r"@Prompt\((?:[^()]*|\([^()]*\))*\)", "'PROMPT_PLACEHOLDER'", sql)
    # Replace @Variable(...) — handles nested parens
    sql = re.sub(r"@Variable\((?:[^()]*|\([^()]*\))*\)", "'VAR_PLACEHOLDER'", sql)
    return sql

def extract_tables(sql):
    try:
        # Pre-process to remove SAP BO-specific syntax
        clean_sql = clean_bo_sql(sql or "")
        tables = set()
        cte_names = set()
        for parsed in sqlglot.parse(clean_sql, read="oracle"):
            # Collect CTE names to exclude them from table references
            for cte in parsed.find_all(CTE):
                cte_names.add(cte.alias.lower())
            # Collect table references
            for t in parsed.find_all(Table):
                catalog = t.catalog
                schema = t.db
                name = t.name
                full_name = ".".join(
                    part for part in [catalog, schema, name] if part
                )
                # Exclude CTE names
                if full_name.lower() not in cte_names:
                    tables.add(full_name)
        return sorted(tables)
    except Exception as e:
        print(f"Error parsing SQL: {e}")
        return []

extract_tables_udf = udf(extract_tables, ArrayType(StringType()))

df = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms")
df_with_tables = df.filter(col("Document_Id") == 10503698).withColumn("parsed_tables", extract_tables_udf(col("SQL_Query")))
df_exploded = df_with_tables.withColumn("table", explode(col("parsed_tables")))

# Split table into schema_Name and table_Name columns (handles 1, 2, or 3-part names)
split_col = split(col("table"), r"\.")
df_final = df_exploded.withColumn(
    "sql_table", upper(trim(col("table")))
).withColumn(
    "table_Name", upper(trim(element_at(split_col, -1)))
).withColumn(
    "schema_Name", when(size(split_col) > 1, upper(trim(element_at(split_col, -2))))
)

# Select columns matching target schema: active_webi_source
result = df_final.select(
    col("Document_Id"),
    col("Document_CUID"),
    col("Document_name"),
    col("Full_path"),
    col("lastAuthor").alias("updated_by"),
    col("Connection_Name").alias("source_DB_connection"),
    col("sql_table"),
    col("table_Name"),
    col("schema_Name")
).distinct()

display(result)

In [0]:
import sqlglot
from sqlglot.expressions import Table
from pyspark.sql.functions import udf, col, explode, split, when, size, element_at, trim, upper
from pyspark.sql.types import ArrayType, StringType
from functools import reduce
from pyspark.sql import DataFrame

def extract_tables(sql):
    try:
        tables = set()
        for parsed in sqlglot.parse(sql or "", read="oracle"):
            for t in parsed.find_all(Table):
                catalog = t.catalog
                schema = t.db
                name = t.name
                full_name = ".".join(
                    part for part in [catalog, schema, name] if part
                )
                tables.add(full_name)
        return sorted(tables)
    except Exception as e:
        print(f"Error parsing SQL: {e}")
        return []

extract_tables_udf = udf(extract_tables, ArrayType(StringType()))

def get_active_webi_source(document_id):
    """Extract table references from SQL_Query for a given Document_Id
    and return a DataFrame matching the active_webi_source schema."""
    df = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms")

    df_with_tables = (
        df.filter(col("Document_Id") == document_id)
          .withColumn("parsed_tables", extract_tables_udf(col("SQL_Query")))
    )

    df_exploded = df_with_tables.withColumn("table", explode(col("parsed_tables")))

    # Split table into schema_Name and table_Name (handles 1, 2, or 3-part names)
    split_col = split(col("table"), r"\.")
    df_final = df_exploded.withColumn(
        "sql_table", upper(trim(col("table")))
    ).withColumn(
        "table_Name", upper(trim(element_at(split_col, -1)))
    ).withColumn(
        "schema_Name", when(size(split_col) > 1, upper(trim(element_at(split_col, -2))))
    )

    # Select columns matching target schema: active_webi_source
    result = df_final.select(
        col("Document_Id"),
        col("Document_CUID"),
        col("Document_name"),
        col("Full_path"),
        col("lastAuthor").alias("updated_by"),
        col("Connection_Name").alias("source_DB_connection"),
        col("sql_table"),
        col("table_Name"),
        col("schema_Name")
    ).distinct()

    return result

# Get Document_Ids not yet processed in active_webi_source
ids_df = spark.sql("""
    SELECT DISTINCT cms.Document_Id 
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms AS cms
    LEFT ANTI JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source AS t
    ON cms.Document_Id = t.Document_Id
    ORDER BY cms.Document_Id ASC
    LIMIT 1000
""")

document_ids = [row.Document_Id for row in ids_df.collect()]
print(f"Processing {len(document_ids)} documents...")

# Loop through each Document_Id and union results
results = []
for i, doc_id in enumerate(document_ids):
    df_result = get_active_webi_source(doc_id)
    results.append(df_result)
    # if (i + 1) % 10 == 0:
    #     print(f"  Processed {i + 1}/{len(document_ids)} documents")

if results:
    final_df = reduce(DataFrame.unionByName, results)
    # Write to target table
    final_df.write.mode("append").saveAsTable(
        "dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source"
    )
    print(f"Successfully written to active_webi_source")
else:
    print("No new documents to process.")

In [0]:
import sqlglot
from sqlglot.expressions import Table
from pyspark.sql.functions import udf, col, explode, split, when, size, element_at, trim, upper
from pyspark.sql.types import ArrayType, StringType
from functools import reduce
from pyspark.sql import DataFrame

def extract_tables(sql):
    try:
        tables = set()
        for parsed in sqlglot.parse(sql or "", read="oracle"):
            for t in parsed.find_all(Table):
                catalog = t.catalog
                schema = t.db
                name = t.name
                full_name = ".".join(
                    part for part in [catalog, schema, name] if part
                )
                tables.add(full_name)
        return sorted(tables)
    except Exception as e:
        print(f"Error parsing SQL: {e}")
        return []

extract_tables_udf = udf(extract_tables, ArrayType(StringType()))

def get_active_webi_source(document_id):
    """Extract table references from SQL_Query for a given Document_Id
    and return a DataFrame matching the active_webi_source schema."""
    df = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms")

    df_with_tables = (
        df.filter(col("Document_Id") == document_id)
          .withColumn("parsed_tables", extract_tables_udf(col("SQL_Query")))
    )

    df_exploded = df_with_tables.withColumn("table", explode(col("parsed_tables")))

    # Split table into schema_Name and table_Name (handles 1, 2, or 3-part names)
    split_col = split(col("table"), r"\.")
    df_final = df_exploded.withColumn(
        "sql_table", upper(trim(col("table")))
    ).withColumn(
        "table_Name", upper(trim(element_at(split_col, -1)))
    ).withColumn(
        "schema_Name", when(size(split_col) > 1, upper(trim(element_at(split_col, -2))))
    )

    # Select columns matching target schema: active_webi_source
    result = df_final.select(
        col("Document_Id"),
        col("Document_CUID"),
        col("Document_name"),
        col("Full_path"),
        col("lastAuthor").alias("updated_by"),
        col("Connection_Name").alias("source_DB_connection"),
        col("sql_table"),
        col("table_Name"),
        col("schema_Name")
    ).distinct()

    return result

# Get Document_Ids not yet processed in active_webi_source
ids_df = spark.sql("""
    SELECT DISTINCT cms.Document_Id 
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms AS cms
    LEFT ANTI JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source AS t
    ON cms.Document_Id = t.Document_Id
    ORDER BY cms.Document_Id ASC
    LIMIT 1000
""")

document_ids = [row.Document_Id for row in ids_df.collect()]
print(f"Processing {len(document_ids)} documents...")

# Loop through each Document_Id and union results
results = []
for i, doc_id in enumerate(document_ids):
    df_result = get_active_webi_source(doc_id)
    results.append(df_result)
    # if (i + 1) % 10 == 0:
    #     print(f"  Processed {i + 1}/{len(document_ids)} documents")

if results:
    final_df = reduce(DataFrame.unionByName, results)
    # Write to target table
    final_df.write.mode("append").saveAsTable(
        "dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source"
    )
    print(f"Successfully written to active_webi_source")
else:
    print("No new documents to process.")

In [0]:
import sqlglot
import re
from sqlglot.expressions import Table, CTE
from sqlglot.errors import ErrorLevel
from pyspark.sql.functions import udf, col, explode, split, when, size, element_at, trim, upper, broadcast, pandas_udf
from pyspark.sql.types import ArrayType, StringType
import pandas as pd


def clean_bo_sql(sql):
    """Remove SAP BO-specific syntax that sqlglot cannot parse."""
    sql = re.sub(r"@Prompt\((?:[^()]*|\([^()]*\))*\)", "'PROMPT_PLACEHOLDER'", sql)
    sql = re.sub(r"@Variable\((?:[^()]*|\([^()]*\))*\)", "'VAR_PLACEHOLDER'", sql)
    sql = re.sub(r'\.\s+', '.', sql)
    return sql

def extract_tables(sql):
    try:
        clean_sql = clean_bo_sql(sql or "")
        tables = set()
        cte_names = set()
        for parsed in sqlglot.parse(clean_sql, read="oracle", error_level=ErrorLevel.IGNORE):
            for cte in parsed.find_all(CTE):
                cte_names.add(cte.alias.lower())
            for t in parsed.find_all(Table):
                catalog = t.catalog
                schema = t.db
                name = t.name
                full_name = ".".join(
                    part for part in [catalog, schema, name] if part
                )
                if full_name.lower() not in cte_names:
                    tables.add(full_name)
        return sorted(tables)
    except Exception as e:
        return []

@pandas_udf(ArrayType(StringType()))
def extract_tables_udf(sql_series: pd.Series) -> pd.Series:
    return sql_series.fillna("").apply(extract_tables)

# Step 1: Get Document_Ids not yet processed
ids_df = spark.sql("""
    SELECT DISTINCT cms.Document_Id 
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms AS cms
    LEFT ANTI JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source AS t
    ON cms.Document_Id = t.Document_Id
    WHERE cms.SQL_Query NOT IN ('Error retrieving Query Plan', 'Data Source Type excel not handled for SQL extraction')
    LIMIT 100
""")

print(f"Documents to process: {ids_df.count()}")

# Step 2: Broadcast join (small ids_df against large source table)
df_filtered = (
    spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms")
         .join(broadcast(ids_df), "Document_Id", "inner")
)

# Step 3: Apply Pandas UDF (vectorized, processes Arrow batches)
df_with_tables = df_filtered.withColumn("parsed_tables", extract_tables_udf(col("SQL_Query")))
df_exploded = df_with_tables.withColumn("table", explode(col("parsed_tables")))

# Step 4: Split and uppercase
split_col = split(col("table"), r"\.")
df_final = df_exploded.withColumn(
    "sql_table", upper(trim(col("table")))
).withColumn(
    "table_Name", upper(trim(element_at(split_col, -1)))
).withColumn(
    "schema_Name", when(size(split_col) > 1, upper(trim(element_at(split_col, -2))))
)

# Step 5: Select target schema, coalesce to reduce output files, and write
final_df = df_final.select(
    col("Document_Id"),
    col("Document_CUID"),
    col("Document_name"),
    col("Full_path"),
    col("lastAuthor").alias("updated_by"),
    col("Connection_Name").alias("source_DB_connection"),
    col("sql_table"),
    col("table_Name"),
    col("schema_Name")
).distinct()

final_df.coalesce(2).write.mode("append").saveAsTable(
    "dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source"
)

print("Done. Batch written to active_webi_source.")